## Informacje o danych

In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/sylwi/OneDrive/Dokumenty/GitHub/project-telco-churn/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.shape)
print(30*"--")
df.info()
print(30*"--")
df.head()

(7043, 21)
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-n

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print(df["Churn"].value_counts())
print(30*"--")
print(df["Churn"].value_counts(normalize=True).round(3))

Churn
No     5174
Yes    1869
Name: count, dtype: int64
------------------------------------------------------------
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


Churn: 26,5% klientów odeszło. Klasy niezbalansowane, więc accuracy nie będzie dobrą metryką.

In [3]:
print(df["customerID"].nunique()) # ile unikalnych klientów (czy nie ma duplikatów?)

bad = df[pd.to_numeric(df["TotalCharges"], errors = "coerce"). isna()] # wartości TotalCharges, które nie dają się zamienić na liczbę
print(len(bad))

bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

7043
11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


TotalCharges trzeba wypełnić zerami. Nie ma sensu usuwać, ponieważ usuniemy dodatkowe dane, a zero to również informacja o statusie klienta.

## Czyszczenie  danych

1. W TotalCharges puste miejsca stają się NaN
2. Te 11 braków to klienci z tenure = 0, czyli łącznie zapłacili 0.
3. customerID to numer, nie cecha
4. Target: Yes/No = 1/0 (1 = odszedł)

In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"] = df["TotalCharges"].fillna(0)

df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [6]:
print(df.shape)
print(30*"--")
print(df["TotalCharges"].dtype, df["TotalCharges"].isna().sum())
print(30*"--")
print(df["Churn"].value_counts())
print(30*"--")
print(df.isna().sum().sum())

(7043, 20)
------------------------------------------------------------
float64 0
------------------------------------------------------------
Churn
0    5174
1    1869
Name: count, dtype: int64
------------------------------------------------------------
0


In [7]:
df.to_csv("../data/processed/telco_clean.csv", index=False)